# Heritage painting restoration – Colab
Runtime → Change runtime type → **T4 GPU** (needed only for training).

In [ ]:
!nvidia-smi | head -5

## 1. Mount Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# >>> change this if your 'images' folder is somewhere else in My Drive <<<
DRIVE_IMAGES = '/content/drive/MyDrive/images'
import os; print(os.listdir(DRIVE_IMAGES))   # should show the 6 period folders

In [ ]:
!git clone https://github.com/cocozero003/historical-painting-restoration-AI.git /content/repo
%cd /content/repo

## 2. Add my patch files
Run this, then pick **sl-restoration-patch.zip** (the zip I sent you).

In [ ]:
from google.colab import files
up = files.upload()
!unzip -o sl-restoration-patch.zip -d /content/repo
!pip -q install -r requirements.txt

## 3. Copy images from Drive (keeps Period/Site folders)

In [ ]:
!mkdir -p data/raw
!cp -r "{DRIVE_IMAGES}"/. data/raw/
!find data/raw -type f | wc -l

## 4. Index + baseline + synthetic training pairs

In [ ]:
!python scripts/index_dataset.py
!python scripts/synthetic_damage_sl.py --split val   --num_per_image 1
!python scripts/evaluate_baseline.py --split val
!python scripts/synthetic_damage_sl.py --split train --num_per_image 3

## 5. (Optional) Train the U-Net crack detector, then save weights to Drive

In [ ]:
!mkdir -p models/unet
!python scripts/train_unet_detector.py --images data/synthetic_damaged --masks data/synthetic_masks --out models/unet/damage_mask_unet.pth
!mkdir -p /content/drive/MyDrive/restoration_outputs
!cp models/unet/damage_mask_unet.pth /content/drive/MyDrive/restoration_outputs/

In [ ]:
# LaMa (better filling for wide cracks) installs with requirements.txt and
# downloads its own weights the first time you tick 'Use LaMa' in the app.
!pip -q install simple-lama-inpainting

## 6. Launch the app (public link appears below)

In [ ]:
from app import demo
demo.launch(share=True, debug=False)